# Load data

In [129]:
import os

import numpy as np
from skimage.io import imread
import cv2 as cv
import numpy as np

import pandas as pd
from typing import List, Tuple, Callable, Optional
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [130]:
def load_data(base_dir, class_names, newSize):
    train_imgs = []
    labels = []

    for class_idx, class_name in enumerate(class_names):
        dir_name = base_dir + class_name
        for filename in os.listdir(dir_name):
            full_path = os.path.join(dir_name, filename)
            img = imread(full_path)#zwraca ndarry postaci xSize x ySize x colorDepth
            img = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
            img = cv.resize(img, newSize, interpolation=cv.INTER_AREA)# zwraca ndarray
            img = img / 255#normalizacja
            train_imgs.append(img)
            labels.append(class_idx)

    return np.array(train_imgs), np.array(labels)

In [131]:
def split_into_train_test(X, y, test_size=0.2, random_state=42):
    rand_indices = np.random.permutation(len(X))
    X = X[rand_indices]
    y = y[rand_indices]

    X = np.array([img.flatten() for img in X])
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

In [132]:
base_dir = "../plantvillage dataset/grayscale/"

In [133]:
newSize = (64, 64)

# Binary classification model

In [134]:
def sigmoid(z):
    """Return the logistic function sigma(z) = 1/(1+exp(-z))."""
    return 1 / (1+np.exp(-z))

def cost(Y, Yhat):
    """Return the cost function for predictions Yhat of classifications Y."""
    return (- Y @ np.log(Yhat.T) - (1 - Y) @ np.log(1 - Yhat.T)) / Y.shape[1]

def accuracy(Y, Yhat):
    """Return measure of the accuracy with which Yhat predicts Y."""
    return 1 - np.mean(np.abs(Y - Yhat.round()))

def model(X, w, b):
    """Apply the logistic model parameterized by w, b to features X."""
    z = w.T @ X + b
    return sigmoid(z)


def train(X, Y, alpha, max_it=1000, debug=False):
    """Train the logistic regression algorithm on the data X classified as Y."""

    # Parameter vector, w, and constant term (bias), b.
    # For random initialization, use the following:
    #w, b = np.random.random((nx,1)) * 0.01, 0.01
    # To initialize with zeros, use this line instead:
    nx = X.shape[0]
    m  = X.shape[1]
    print(f"m={m}, nx={nx}")

    w, b = np.zeros((nx,1)), 0

    def propagate(w, b):
        """Propagate the training by advancing w, b to reduce the cost, J."""
        Yhat = model(X, w, b)
        w -= alpha / m * (X @ (Yhat - Y).T)
        b -= alpha / m * np.sum(Yhat - Y)
        J = np.squeeze(cost(Y, Yhat))
        if debug and not it % 100:
            # Provide an update on the progress we have made so far.
            print('{}: J = {}'.format(it, J))
            print('train accuracy = {:g}%'.format(accuracy(Y, Yhat) * 100))
        return w, b

    # Train the model by iteratively improving w, b.
    for it in range(max_it):
        w, b = propagate(w, b)
    return w, b


def predict(X, w, b):
    return model(X, w, b).round().flatten()

In [135]:
def eval(w, b, X_test, Y_expected):
    Y_predicted = predict(X_test, w, b)

    tp = 0
    tn = 0
    fp = 0
    fn = 0

    for i in range(len(Y_expected)):
        if Y_expected[i] == 1 and Y_predicted[i] == 1:
            tp += 1
        elif Y_expected[i] == 0 and Y_predicted[i] == 0:
            tn += 1
        elif Y_expected[i] == 0 and Y_predicted[i] == 1:
            fp += 1
        elif Y_expected[i] == 1 and Y_predicted[i] == 0:
            fn += 1

    accuracy = (tp + tn) / (tp + tn + fp + fn)
    print('Accuracy:', accuracy)

    precision = tp / (tp + fp)
    print('Precision:', precision)

    recall = tp / (tp + fn)
    print('Recall:', recall)

    fscore = (2 * precision * recall) / (precision + recall)
    print('F-score:', fscore)

# Blueberry___healthy vs Grape___healthy

In [136]:
X, y = load_data(base_dir, ["Blueberry___healthy", "Grape___healthy"], newSize)
X_train, X_test, y_train, y_test = split_into_train_test(X, y, test_size=0.2, random_state=42)

In [137]:
cw, cb = train(X_train.T, y_train.reshape((1, y_train.shape[0])),
               alpha=0.01, max_it=5_000)

m=1540, nx=4096


In [138]:
eval(cw, cb, X_test.T, y_test.T)

Accuracy: 0.8753246753246753
Precision: 0.7164179104477612
Recall: 0.6233766233766234
F-score: 0.6666666666666666


# Blueberry___healthy vs Strawberry___healthy

In [139]:
X, y = load_data(base_dir, ["Blueberry___healthy", "Strawberry___healthy"], newSize)
X_train, X_test, y_train, y_test = split_into_train_test(X, y, test_size=0.2, random_state=42)

In [140]:
cw, cb = train(X_train.T, y_train.reshape((1, y_train.shape[0])),
               alpha=0.01, max_it=5_000)

m=1566, nx=4096


In [141]:
eval(cw, cb, X_test.T, y_test.T)

Accuracy: 0.8418367346938775
Precision: 0.7083333333333334
Recall: 0.5543478260869565
F-score: 0.6219512195121951


# Grape___healthy vs Strawberry___healthy

In [142]:
X, y = load_data(base_dir, ["Grape___healthy", "Strawberry___healthy"], newSize)
X_train, X_test, y_train, y_test = split_into_train_test(X, y, test_size=0.2, random_state=42)

In [143]:
cw, cb = train(X_train.T, y_train.reshape((1, y_train.shape[0])),
               alpha=0.01, max_it=5_000)

m=703, nx=4096


In [144]:
eval(cw, cb, X_test.T, y_test.T)

Accuracy: 0.8181818181818182
Precision: 0.8452380952380952
Recall: 0.7888888888888889
F-score: 0.8160919540229884
